# Hypothesis Testing

## 1. Hypothesis Definition

This section aims to investigate whether productivity exhibits a statistically meaningful linear association with caffeine intake and sleep pattern. 

- **Null Hypothesis (H₀):** There is no significant relationship between my productivity and my daily caffeine consumption or sleep pattern. 
- **Alternative Hypothesis (H₁):** There is a significant relationship between my productivity and my daily caffeine consumption or sleep pattern. 

The hypotheses will be evaluated through Pearson correlation analysis, focusing on caffeine consumption and sleep pattern. Evidence of statistically significant associations may lay the groundwork for developing machine learning models that predict productivity using behavioral patterns.


### 2. Loading and Preparing Data

As in the Exploratory Data Analysis section, we start by loading the data, and refining it (removing unnecessary parts and cleaning it), so we can use the data more effectively for analysis and testing our hypotheses. 



In [ ]:

# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read the dataset
df = pd.read_csv('datasheet.csv')
# Delete all empty columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
# Rename columns for clarity
df.columns = [
    'DATE', 'TotalSleep', 'TotalSleep_min', 'REM', 'REM_min',
    'Light', 'Light_min', 'Deep', 'Deep_min', 'Caffeine_mg',
    'Productivity', 'Productivity_min'
]

# Convert Caffeine to numeric
df['Caffeine_mg'] = df['Caffeine_mg'].str.replace('mg','').str.strip()
df['Caffeine_mg'] = pd.to_numeric(df['Caffeine_mg'], errors='coerce')

# Convert DATE to datetime format and sort
df['DATE'] = pd.to_datetime(df['DATE'] + ' 2025', format='%B %d %Y', errors='coerce')
df.sort_values('DATE', inplace=True)



                  count    mean         std    min    25%    50%    75%    max
Caffeine_mg        25.0  114.76  113.518457    0.0   25.0  135.0  135.0  410.0
Productivity_min   25.0  196.60  100.485488   60.0  105.0  180.0  300.0  360.0
TotalSleep_min     25.0  438.60   52.605450  324.0  412.0  450.0  468.0  557.0
REM_min            25.0   95.28   23.129202   49.0   85.0   95.0  107.0  145.0
Light_min          25.0  288.68   42.891064  191.0  271.0  290.0  317.0  379.0
Deep_min           25.0   51.56   12.662543   33.0   39.0   52.0   59.0   75.0



### 3. Visualization Before Testing

Prior to conducting the statistical analysis, it is helpful to visually examine the relationship between productivity and both caffeine intake and sleep duration, just as we did in the EDA. This allows us to evaluate the validity of using a linear test like Pearson correlation. 



In [ ]:
# Caffeine vs Productivity scatter plot
plt.figure(figsize=(8,5))
sns.scatterplot(x='Caffeine_mg', y='Productivity_min', data=df)
plt.title('Caffeine vs Productivity')
plt.xlabel('Caffeine (mg)')
plt.ylabel('Productivity (minutes)')
plt.show()

# Total sleep vs Productivity scatter plot 
plt.figure(figsize=(8,5))
sns.scatterplot(x='TotalSleep_min', y='Productivity_min', data=df)
plt.title('Total Sleep vs Productivity')
plt.xlabel('Total Sleep (minutes)')
plt.ylabel('Productivity (minutes)')
plt.show()

# Productivity vs Caffeine Intake Over Time
def normalize(col): #normalizing values to visually compare the 2 variables
    return (col - col.min()) / (col.max() - col.min())
plt.figure(figsize=(10,5))
plt.plot(df['DATE'], normalize(df['Productivity_min']), label='Productivity')
plt.plot(df['DATE'], normalize(df['Caffeine_mg']), label='Caffeine Intake', color='red')
plt.title('Productivity vs Caffeine Intake Over Time')
plt.xlabel('Date')
plt.ylabel('Normalized Scale (0–1)')
plt.xticks(rotation=45)
plt.legend()
plt.show()

# Productivity vs Total Sleep Time Over Time
plt.figure(figsize=(10,5))
plt.plot(df['DATE'], normalize(df['Productivity_min']), label='Productivity')
plt.plot(df['DATE'], normalize(df['TotalSleep_min']), label='Total Sleep Time', color='yellowgreen')
plt.title('Productivity vs Total Sleep Time Over Time')
plt.xlabel('Date')
plt.ylabel('Normalized Scale (0–1)')
plt.xticks(rotation=45)
plt.legend()
plt.show()

# CORRELATION MATRIX
all_corr = df[['TotalSleep_min', 'REM_min', 'Light_min', 'Deep_min', 'Caffeine_mg', 'Productivity_min']].corr()
print("Correlation Matrix for All Variables:\n", all_corr)
plt.figure(figsize=(8,6))
sns.heatmap(all_corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix: Sleep, Caffeine & Productivity')
plt.show()


![Caffeine vs Productivity scatter plot](figures/caffeine_vs_productivity.jpg)



![Total sleep vs Productivity scatter plot](figures/totalsleep_vs_productivity.jpg)



![Productivity vs Caffeine Intake Over Time](figures/productivity_vs_caffeine_ot.jpg)



![Productivity vs Total Sleep Time Over Time](figures/productivity_vs_totalsleep_ot.jpg)



![Correlation Matrix](figures/correlationmatrix.jpg)



### 4. Pearson Correlation Test

We will calculate the Pearson correlation coefficient **r** along with the corresponding **p-value** to assess how productivity relates to caffeine consumption and, separately, to sleep pattern.



In [ ]:

from scipy.stats import pearsonr

variables = ['Caffeine_mg', 'TotalSleep_min']

print("\nPearson Correlation Results\n")
print(f"{'Variable':<20} {'r-value':<10} {'p-value':<10} Decision")
print('-' * 51)

for var in variables:
    r, p = pearsonr(df[var], df['Productivity_min'])
    decision = 'Reject H₀' if p < 0.05 else 'Fail to Reject H₀'
    print(f"{var:<20} {r:.3f}      {p:.4f}    {decision}")



Pearson Correlation Results
Variable             r-value    p-value    Decision
---------------------------------------------------
Caffeine_mg          0.830      0.0000    Reject H₀
TotalSleep_min       0.551      0.0043    Reject H₀



### 5. Interpretation

The Pearson correlation test reveals statistically significant relationships between the productivity and both factors examined.

For the **caffeine consumption**, the positive correlation coefficient is r = 0.830 with p-value < 0.0001 (≈ 0.0000). Since the p-value is below the significance level α = 0.05, the **null hypothesis is rejected**. This indicates a **strong positive linear association between caffeine consumption and productivity**: as caffeine intake increases, productivity tends to rise proportionally in this dataset. 

For the sleep duration, the positive correlation coefficient is r = 0.551 with p-value = 0.0043. The p-value of sleep duration is also below the signifance level 0.05, so we **reject the null hypothesis** again. This results suggests a **moderate positive linear association between total sleep time and productivity**, meaning higher sleep duration is generally linked with higher productivity, though the association is not as strong as that observed for caffeine.

Overall, the results support the alternative hypothesis, indicating that there is a significant relationship between my productivity and both my daily caffeine consumption and sleep patterns.

